In [1]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")  # Update with your MongoDB URI if needed
db = client["cve_db"]  # Select database
collection = db["cves"]  # Select collection

print("Connected to MongoDB!")

Connected to MongoDB!


In [ ]:
import pandas as pd

# Load data from MongoDB into a DataFrame
cve_data = list(collection.find({}))
df = pd.DataFrame(cve_data)

# Display first 5 rows
df.head()

In [ ]:
# Remove _id column (MongoDB-specific)
df.drop(columns=["_id"], inplace=True)

# Trim whitespace in string columns
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Fill missing descriptions with "No description available"
df["description"].fillna("No description available", inplace=True)

# Fill missing severity with "Unknown"
df["severity"].fillna("Unknown", inplace=True)

# Show cleaned data
df.head()

In [ ]:
# Convert cleaned DataFrame back to dictionary format
cleaned_cve_data = df.to_dict(orient="records")

# Replace data in MongoDB
collection.delete_many({})  # Clear existing data
collection.insert_many(cleaned_cve_data)

print(f"Cleaned {len(cleaned_cve_data)} CVEs and updated MongoDB!")

In [ ]:
# Fetch and display a few cleaned records
cleaned_data = list(collection.find({}))
pd.DataFrame(cleaned_data).head()